In [12]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append(r"C:\Users\dillo\OneDrive\Documents\Projects\Microsoft Mind")
import numpy as np, pandas as pd
from collections import Counter
from scipy import stats
from src.db import get_engine
from src.eval import ndcg_at_k, mrr
from src.metrics import intra_list_diversity
from src.recommenders import content_recommender
from src.rerank import make_reranker, evaluate_ranker
engine = get_engine()

emb = np.load("../models/news_emb.npy")
news_ids = np.load("../models/news_emb_ids.npy", allow_pickle=True)
row_of = {n: i for i, n in enumerate(news_ids)}

train_clicks = pd.read_sql("SELECT news_id, SUM(clicked) AS clicks FROM fact_impressions GROUP BY news_id",
                           engine).set_index("news_id")["clicks"].to_dict()
meta = pd.read_sql("SELECT news_id, category, subcategory FROM dim_news", engine)
cat_of    = dict(zip(meta.news_id, meta.category))
subcat_of = dict(zip(meta.news_id, meta.subcategory))

def parse(s): return [(nid, int(l)) for nid, l in (x.split('-') for x in s.split())]
def profile_from_history(h):
    if not isinstance(h, str) or not h.strip(): return None
    c, s = Counter(), Counter()
    for nid in h.split():
        if nid in cat_of:    c[cat_of[nid]] += 1
        if nid in subcat_of: s[subcat_of[nid]] += 1
    t = sum(c.values())
    return {"cats": c, "subs": s, "total": t} if t else None

dev = pd.read_csv("../data/raw/MINDsmall_dev/behaviors.tsv", sep="\t",
       names=["impression_id","user_id","time","history","impressions"])
dev["parsed"]  = dev["impressions"].apply(parse)
dev["profile"] = dev["history"].apply(profile_from_history)

base_score     = content_recommender(cat_of, subcat_of, train_clicks)
control_ranker = make_reranker(base_score, emb, row_of, lam=0.0)
treat_ranker   = make_reranker(base_score, emb, row_of, lam=0.5)

fs = pd.read_sql("SELECT news_id, MIN(shown_at) AS first_seen "
                 "FROM fact_impressions GROUP BY news_id", engine)
first_seen = dict(zip(fs.news_id, pd.to_datetime(fs.first_seen)))

dev["shown_at"] = pd.to_datetime(dev["time"], format="%m/%d/%Y %I:%M:%S %p")

for row in dev.itertuples(index=False):
    for nid, _ in row.parsed:
        if nid not in first_seen or row.shown_at < first_seen[nid]:
            first_seen[nid] = row.shown_at

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
rng = np.random.default_rng(42)
users = dev["user_id"].unique()
sample_users = rng.choice(users, size=8000, replace=False)
arm = {u: ("treatment" if rng.random() < 0.5 else "control") for u in sample_users}

exp = dev[dev["user_id"].isin(sample_users)].copy()
exp["arm"] = exp["user_id"].map(arm)

from collections import Counter, defaultdict
from src.metrics import (intra_list_diversity, repetition_rate, freshness,
                         category_fairness, overexposure_ratio,
                         fatigue_per_user, fatigue_index)

def run_experiment(exp, k=10):
    recs = []
    exposure = {"control": Counter(), "treatment": Counter()}
    demand   = {"control": Counter(), "treatment": Counter()}
    user_cat = {"control": defaultdict(Counter), "treatment": defaultdict(Counter)}

    for row in exp.itertuples(index=False):
        parsed = row.parsed
        if sum(l for _, l in parsed) == 0:
            continue
        arm    = row.arm
        ranker = treat_ranker if arm == "treatment" else control_ranker
        order  = ranker(row)                      # full list: ideal DCG needs it
        lab    = dict(parsed)
        labels = [lab[nid] for nid in order]
        top    = order[:k]                        # what the user would actually see

        for nid in top:                   # exposure: only what was actually shown
            c = cat_of.get(nid)
            if c is None:
                continue
            exposure[arm][c] += 1
            user_cat[arm][row.user_id][c] += 1

        for nid, l in parsed:             # demand: all clicks available, ranking-independent
            c = cat_of.get(nid)
            if c is not None:
                demand[arm][c] += l

        recs.append({"user_id": row.user_id, "arm": arm,
                     "ndcg": ndcg_at_k(labels, k), "mrr": mrr(labels),
                     "diversity":  intra_list_diversity(top, emb, row_of),
                     "repetition": repetition_rate(top, emb, row_of, tau=0.9),
                     "freshness":  freshness(top, row.shown_at, first_seen, tau_hours=24)})

    return pd.DataFrame(recs), exposure, demand, user_cat


results, exposure, demand, user_cat = run_experiment(exp)
print(results.groupby("arm")[["ndcg", "mrr", "diversity", "repetition", "freshness"]].mean())

for arm in ("control", "treatment"):
    cat, ratio = overexposure_ratio(exposure[arm], demand[arm])
    print(f"{arm:10s} fairness={category_fairness(exposure[arm]):.3f}  "
          f"fatigue={fatigue_index(user_cat[arm]):.3f}  "
          f"worst over-exposure: {cat} ({ratio:.2f}x demand)")

               ndcg       mrr  diversity  repetition  freshness
arm                                                            
control    0.389354  0.346539   0.911737         0.0   0.534575
treatment  0.375923  0.336518   0.934771         0.0   0.558595
control    fairness=0.393  fatigue=0.524  worst over-exposure: news (1.60x demand)
treatment  fairness=0.398  fatigue=0.517  worst over-exposure: news (1.68x demand)


In [14]:
per_user = (results.groupby(["arm", "user_id"])
            [["ndcg", "diversity", "repetition", "freshness"]].mean().reset_index())

fat = pd.DataFrame([{"arm": a, "user_id": u, "fatigue": v}
                    for a in ("control", "treatment")
                    for u, v in fatigue_per_user(user_cat[a]).items()])
per_user = per_user.merge(fat, on=["arm", "user_id"], how="left")

for metric in ["ndcg", "diversity", "repetition", "freshness", "fatigue"]:
    c = per_user.loc[per_user.arm == "control",   metric].dropna()
    t = per_user.loc[per_user.arm == "treatment", metric].dropna()
    diff = t.mean() - c.mean()
    stat, p = stats.ttest_ind(t, c, equal_var=False)
    se = np.sqrt(t.var()/len(t) + c.var()/len(c))
    print(f"{metric:11s} lift={diff:+.4f}  95% CI=({diff-1.96*se:+.4f},{diff+1.96*se:+.4f})  p={p:.2e}")

ndcg        lift=-0.0094  95% CI=(-0.0232,+0.0044)  p=1.84e-01
diversity   lift=+0.0234  95% CI=(+0.0211,+0.0257)  p=7.01e-85
repetition  lift=+0.0000  95% CI=(+0.0000,+0.0000)  p=nan
freshness   lift=+0.0243  95% CI=(+0.0203,+0.0284)  p=2.16e-31
fatigue     lift=-0.0075  95% CI=(-0.0175,+0.0026)  p=1.45e-01


In [15]:
from collections import defaultdict

rng = np.random.default_rng(0)
by_cat = defaultdict(list)
for nid, i in row_of.items():
    c = cat_of.get(nid)
    if c is not None:
        by_cat[c].append(i)

rows = []
for c, idx in by_cat.items():
    if len(idx) < 50:
        continue
    s = rng.choice(idx, size=min(500, len(idx)), replace=False)
    V = emb[s]
    S = V @ V.T
    n = len(s)
    rows.append({"category": c, "n_articles": len(idx),
                 "dispersion": 1 - (S.sum() - n) / (n * (n - 1))})
disp = pd.DataFrame(rows)

tot_c, tot_t = sum(exposure["control"].values()), sum(exposure["treatment"].values())
share = pd.DataFrame({"category": sorted(set(exposure["control"]) | set(exposure["treatment"]))})
share["control_share"]   = share.category.map(lambda c: exposure["control"][c] / tot_c)
share["treatment_share"] = share.category.map(lambda c: exposure["treatment"][c] / tot_t)
share["delta"] = share.treatment_share - share.control_share

m = share.merge(disp, on="category", how="inner")
print(m.sort_values("delta", ascending=False).round(4).to_string(index=False))
print("\ncorr(exposure shift, within-category dispersion) =",
      round(m["delta"].corr(m["dispersion"]), 3))

     category  control_share  treatment_share   delta  n_articles  dispersion
         news         0.3752           0.3903  0.0151       15774      0.9130
      weather         0.0161           0.0252  0.0091        2048      0.8037
        autos         0.0175           0.0221  0.0046        1639      0.8417
       movies         0.0186           0.0225  0.0038         606      0.8489
       travel         0.0108           0.0124  0.0016        2350      0.8969
       health         0.0238           0.0254  0.0015        1885      0.9150
      finance         0.0437           0.0444  0.0007        3107      0.9086
        video         0.0072           0.0064 -0.0008        2068      0.9052
        music         0.0224           0.0199 -0.0025         769      0.8518
entertainment         0.0180           0.0145 -0.0035         587      0.8739
           tv         0.0509           0.0465 -0.0044         889      0.8706
    lifestyle         0.1553           0.1491 -0.0062        247

In [ ]:
train_ids = set(pd.read_sql("SELECT DISTINCT news_id FROM fact_impressions", engine).news_id)

paired = dev.sample(n=3000, random_state=42)
added_cat, dropped_cat = Counter(), Counter()
added_total = added_new = 0
d_ndcg, fresh_add, fresh_drop = [], [], []

for row in paired.itertuples(index=False):
    parsed = row.parsed
    if sum(l for _, l in parsed) == 0:
        continue
    lab = dict(parsed)
    oc, ot = control_ranker(row), treat_ranker(row)
    d_ndcg.append(ndcg_at_k([lab[n] for n in ot], 10) - ndcg_at_k([lab[n] for n in oc], 10))

    sc, st = set(oc[:10]), set(ot[:10])
    for nid in st - sc:
        added_total += 1
        added_cat[cat_of.get(nid)] += 1
        added_new += nid not in train_ids
        fresh_add.append(freshness([nid], row.shown_at, first_seen, tau_hours=24))
    for nid in sc - st:
        dropped_cat[cat_of.get(nid)] += 1
        fresh_drop.append(freshness([nid], row.shown_at, first_seen, tau_hours=24))

d = np.array(d_ndcg)
se = d.std(ddof=1) / np.sqrt(len(d))
margin = 0.01 * 0.363          # the ~1% engagement budget, in nDCG units
print(f"paired dNDCG@10 = {d.mean():+.4f}  95% CI=({d.mean()-1.96*se:+.4f}, {d.mean()+1.96*se:+.4f})  n={len(d)}")
print(f"equivalence margin = +/-{margin:.4f} -> "
      f"{'PASSES' if abs(d.mean())+1.96*se < margin else 'INCONCLUSIVE'} at the 1% budget\n")

print(f"slots swapped in: {added_total}; absent from train: {added_new/max(added_total,1):.1%}")
print(f"mean freshness  added={np.nanmean(fresh_add):.3f}  dropped={np.nanmean(fresh_drop):.3f}\n")

swap = pd.DataFrame({"added": pd.Series(added_cat), "dropped": pd.Series(dropped_cat)}).fillna(0)
swap["net"] = swap["added"] - swap["dropped"]
print(swap.sort_values("net", ascending=False).to_string())

paired dNDCG@10 = -0.0072  95% CI=(-0.0108, -0.0036)  n=3000
equivalence margin = +/-0.0036 -> INCONCLUSIVE at the 1% budget

slots swapped in: 4281; absent from train: 55.7%
mean freshness  added=0.623  dropped=0.433

               added  dropped  net
news            1639     1217  422
weather          308        2  306
autos            191       81  110
movies           190       91   99
health           143      103   40
finance          286      262   24
video             52       35   17
travel            79       70    9
music             41      120  -79
tv               169      259  -90
entertainment     39      204 -165
foodanddrink     192      373 -181
sports           536      740 -204
lifestyle        416      724 -308


In [17]:
cand = {nid for row in dev.itertuples(index=False) for nid, _ in row.parsed}
no_cat = {n for n in cand if n not in cat_of}
print(f"dev candidates: {len(cand)}; missing from dim_news: {len(no_cat)} ({len(no_cat)/len(cand):.1%})")
print(f"  of those, also missing an embedding: {sum(n not in row_of for n in no_cat)}")

dev candidates: 5369; missing from dim_news: 0 (0.0%)
  of those, also missing an embedding: 0


In [18]:
cols = ['news_id','category','subcategory','title','abstract','url','title_entities','abstract_entities']
dev_news = pd.read_csv("../data/raw/MINDsmall_dev/news.tsv", sep="\t", names=cols)
existing = set(pd.read_sql("SELECT news_id FROM dim_news", engine).news_id)
new_rows = (dev_news[~dev_news.news_id.isin(existing)]
            .drop_duplicates("news_id")[['news_id','category','subcategory','title','abstract','url']])
new_rows.to_sql("dim_news", engine, if_exists="append", index=False)
print(len(new_rows), "articles added to dim_news")

0 articles added to dim_news


In [19]:
print("dev impressions:", len(dev), "| users:", dev.user_id.nunique())
print("mean candidates/impression:", round(dev.parsed.map(len).mean(), 1))
print("dim_news rows:", pd.read_sql("SELECT COUNT(*) AS c FROM dim_news", engine).c[0])

dev impressions: 73152 | users: 50000
mean candidates/impression: 37.5
dim_news rows: 65238


In [20]:
tn = pd.read_csv("../data/raw/MINDsmall_train/news.tsv", sep="\t", names=cols)
dn = pd.read_csv("../data/raw/MINDsmall_dev/news.tsv",   sep="\t", names=cols)
print("dev-only articles:", len(set(dn.news_id) - set(tn.news_id)))

dev-only articles: 13956


In [21]:
lo, hi = d.mean()-1.96*se, d.mean()+1.96*se
if lo > -margin and hi < margin:      verdict = "EQUIVALENT within budget"
elif hi < -margin:                    verdict = "BREACH: loss exceeds budget"
elif lo > margin:                     verdict = "IMPROVEMENT beyond budget"
else:                                 verdict = "INCONCLUSIVE (CI spans the margin)"
print(f"{verdict}  (margin +/-{margin:.4f})")

BREACH: loss exceeds budget  (margin +/-0.0036)


In [22]:
cent = {}
for c, idx in by_cat.items():
    if len(idx) >= 50:
        v = emb[idx].mean(0); cent[c] = v / np.linalg.norm(v)
G = emb.mean(0); G = G / np.linalg.norm(G)
iso = pd.DataFrame([{"category": c, "isolation": 1 - float(v @ G)} for c, v in cent.items()])

m2 = share.merge(iso, on="category").merge(disp, on="category")
print(m2.sort_values("delta", ascending=False).round(4).to_string(index=False))
print("corr(delta, isolation)  =", round(m2.delta.corr(m2.isolation), 3))
print("corr(delta, dispersion) =", round(m2.delta.corr(m2.dispersion), 3))

     category  control_share  treatment_share   delta  isolation  n_articles  dispersion
         news         0.3752           0.3903  0.0151     0.1569       15774      0.9130
      weather         0.0161           0.0252  0.0091     0.4548        2048      0.8037
        autos         0.0175           0.0221  0.0046     0.4891        1639      0.8417
       movies         0.0186           0.0225  0.0038     0.5016         606      0.8489
       travel         0.0108           0.0124  0.0016     0.3234        2350      0.8969
       health         0.0238           0.0254  0.0015     0.5774        1885      0.9150
      finance         0.0437           0.0444  0.0007     0.4297        3107      0.9086
        video         0.0072           0.0064 -0.0008     0.2226        2068      0.9052
        music         0.0224           0.0199 -0.0025     0.4593         769      0.8518
entertainment         0.0180           0.0145 -0.0035     0.4292         587      0.8739
           tv        

In [23]:
def paired_delta(lam, n_imp=6000, k=10, seed=42):
    base_r = make_reranker(base_score, emb, row_of, lam=0.0)
    treat  = make_reranker(base_score, emb, row_of, lam=lam)
    d_n, d_v = [], []
    for row in dev.sample(n=n_imp, random_state=seed).itertuples(index=False):
        parsed = row.parsed
        if sum(l for _, l in parsed) == 0:
            continue
        lab = dict(parsed)
        oc, ot = base_r(row), treat(row)
        d_n.append(ndcg_at_k([lab[x] for x in ot], k) - ndcg_at_k([lab[x] for x in oc], k))
        d_v.append(intra_list_diversity(ot[:k], emb, row_of) -
                   intra_list_diversity(oc[:k], emb, row_of))
    d_n, d_v = np.array(d_n), np.array(d_v)
    se = d_n.std(ddof=1) / np.sqrt(len(d_n))
    return {"lambda": lam, "dNDCG": d_n.mean(), "lo": d_n.mean()-1.96*se,
            "hi": d_n.mean()+1.96*se, "dDiv": float(np.nanmean(d_v))}

sweep = pd.DataFrame([paired_delta(l) for l in [0.1, 0.2, 0.3, 0.4, 0.5, 0.7]])
sweep["pct_ndcg"]  = 100 * sweep.dNDCG / 0.389
sweep["exchange"]  = sweep.dDiv / -sweep.dNDCG      # diversity bought per unit nDCG paid
print(sweep.round(4).to_string(index=False))

 lambda   dNDCG      lo      hi   dDiv  pct_ndcg  exchange
    0.1 -0.0032 -0.0051 -0.0013 0.0065   -0.8229    2.0460
    0.2 -0.0033 -0.0053 -0.0013 0.0087   -0.8515    2.6389
    0.3 -0.0039 -0.0060 -0.0019 0.0122   -1.0137    3.0835
    0.4 -0.0057 -0.0079 -0.0035 0.0168   -1.4711    2.9438
    0.5 -0.0079 -0.0104 -0.0054 0.0233   -2.0335    2.9476
    0.7 -0.0170 -0.0202 -0.0139 0.0421   -4.3779    2.4731
